# 🏛️ UNIVERSIDAD NACIONAL DE SAN ANTONIO ABAD DEL CUSCO
### Facultad de Ingeniería Civil — Laboratorio de Materiales y Concreto
**Asignatura:** Tecnología de Materiales (IC151)  
**Docente:** ______________________________ | **Grupo / Brigada:** ______________________________  
**Semestre:** 2026-II  

---

# 🔬 PRÁCTICA N.° 1 (TM-01): Tiempo de Fraguado del Yeso (Método del Cuchillo y Vicat)
### Cuaderno Interactivo de Procesamiento y Control Normativo

> **Normativa Técnica Aplicable:**  
> **EN 13279-2** (espolvoreo y cuchillo) | **ASTM C472-99 (R2014) / NTP 239.651:2025** (Vicat 300 g / 1 mm) | **EN 13279-1 / ASTM C28** (límites referenciales)

---

### 🎯 Objetivos de la Práctica
1. Determinar la relación agua/yeso (*R*) mediante el método de espolvoreo (EN 13279-2, num. 4.3.1).
2. Determinar el tiempo de fraguado inicial mediante la prueba de corte con cuchillo (EN 13279-2, num. 4.4.1).
3. Medir el tiempo de fraguado completo con la aguja estándar de Vicat de 300 g y 1 mm (ASTM C472, num. 10), dosificando la pasta con *R* por espolvoreo ante la ausencia del émbolo cónico de 35 g para consistencia normal (ASTM C472, num. 8).
4. Contrastar de manera referencial ambos tiempos frente a los criterios normativos y evaluar la coherencia física (*T_cuchillo < T_Vicat*).

## ⚙️ 1. Conexión con el Motor de Cálculo (`civillab_unsaac`)
Presiona **Ejecutar (▶)** en la siguiente celda antes de comenzar. Conecta las fórmulas normativas de la UNSAAC, tablas y graficadores.

In [1]:
# ==============================================================================
# CONEXIÓN CON LA LIBRERÍA DEL LABORATORIO (Ejecutar primero, no modificar)
# ==============================================================================
try:
    import civillab_unsaac as lab
except ModuleNotFoundError:
    # Si la librería aún no está instalada, la ubicamos automáticamente en el proyecto
    import sys
    from pathlib import Path
    raices = [
        Path.cwd(),
        *list(Path.cwd().parents)[:3],
        Path.cwd() / "08_SOFTWARE_LABORATORIO",
        Path.cwd() / "0003_UNSAAC_DOCENCIA" / "08_SOFTWARE_LABORATORIO",
    ]
    for r in raices:
        if (r / "civillab_unsaac").is_dir():
            sys.path.insert(0, str(r))
            break
    import civillab_unsaac as lab

from pathlib import Path
from civillab_unsaac import *
import matplotlib
try:
    get_ipython()
except NameError:
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    display = print

# Carpeta limpia para gráficos y fichas Excel de salida
_pkg_root = Path(lab.__file__).resolve().parents[1]
DIR_SALIDA = (_pkg_root / "fichas_generadas") if (_pkg_root / "notebooks").is_dir() else (Path.cwd() / "fichas_generadas")
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

print("✓ Suite CivilLab UNSAAC conectada y lista para calcular.")

✓ Suite CivilLab UNSAAC conectada y lista para calcular.


---
## 📊 2. Ingreso de Datos de Laboratorio (Yeso de Construcción)
Registra las masas de los ensayos de espolvoreo (`m₀` y `m₁` sobre 100 g de agua en `120 ± 5 s`), la masa de yeso empleada en Vicat (`200 a 300 g`, sin retardante) y los tiempos cronometrados desde el instante en que el polvo toca el agua.

In [2]:
# Ingreso de datos tomados en la mesa de trabajo:
procedencia_yeso = "Yeso de construcción comercial (sin retardante declarado)"

# Parte A1: Ensayo de espolvoreo sobre 100 g de agua (EN 13279-2, num. 4.3.1)
ensayos_espolvoreo = [
    {"m0_g": 250.0, "m1_g": 403.5, "tiempo_s": 118},  # Ensayo 1
    {"m0_g": 250.2, "m1_g": 404.5, "tiempo_s": 122},  # Ensayo 2
]
r_lista = [100.0 / (e["m1_g"] - e["m0_g"]) for e in ensayos_espolvoreo]
rel_ay = float(sum(r_lista) / len(r_lista))

# Partes A2 y B: Dosificación y tiempos de fraguado (usando R del espolvoreo)
masa_yeso_vicat_g = 250.0     # 200 a 300 g para llenar el anillo de Vicat (ASTM C472, num. 8.3.2 y 10.3.1)
vol_agua_vicat_ml = round(masa_yeso_vicat_g * rel_ay, 1)

tiempo_cuchillo_min = 24.5    # Fraguado inicial: bordes del corte dejan de juntarse (EN 13279-2, num. 4.4.1)
tiempo_vicat_min = 36.0       # Fraguado completo: aguja de 1 mm (300 g) ya no penetra al fondo (ASTM C472, num. 10.3)

# Evaluación normativa y de coherencia física:
espolvoreo_en_tiempo = all(115 <= e["tiempo_s"] <= 125 for e in ensayos_espolvoreo)
cumple_cuchillo = tiempo_cuchillo_min > 20.0
cumple_vicat_ref = 20.0 <= tiempo_vicat_min <= 40.0
coherente = tiempo_cuchillo_min < tiempo_vicat_min

resumen = pd.DataFrame([
    {
        "Ensayo / Parámetro": "Relación agua/yeso por espolvoreo (R)",
        "Valor Medido": f"{rel_ay:.2f} (100 g agua / {100.0 / rel_ay:.1f} g yeso)",
        "Criterio Normado": "Espolvoreo 120 ± 5 s (EN 13279-2, 4.3.1)",
        "Dictamen": "CUMPLE TIEMPO" if espolvoreo_en_tiempo else "FUERA DE 120 ± 5 s",
    },
    {
        "Ensayo / Parámetro": "Fraguado Inicial (Cuchillo EN 13279-2)",
        "Valor Medido": f"{tiempo_cuchillo_min:.1f} min",
        "Criterio Normado": "> 20.0 min (EN 13279-1, B7 ref.)",
        "Dictamen": "CONFORME" if cumple_cuchillo else "MENOR A 20 min (DECLARAR FABRICANTE)",
    },
    {
        "Ensayo / Parámetro": "Fraguado Completo (Vicat 300 g / 1 mm)",
        "Valor Medido": f"{tiempo_vicat_min:.1f} min",
        "Criterio Normado": "20.0 a 40.0 min (ASTM C28, referencial sin émbolo 35 g)",
        "Dictamen": (
            ("CONFORME REF." if cumple_vicat_ref else "FUERA DE RANGO REF.")
            + ("" if coherente else " [INCOHERENTE < CUCHILLO]")
        ),
    },
])

print(f"Material: {procedencia_yeso}")
print(
    f"Parte B (Vicat 300 g / 1 mm, sin retardante): "
    f"{masa_yeso_vicat_g:.0f} g yeso + {vol_agua_vicat_ml:.1f} mL agua (con R = {rel_ay:.2f})"
)
display(resumen)

Material: Yeso de construcción comercial (sin retardante declarado)
Parte B (Vicat 300 g / 1 mm, sin retardante): 250 g yeso + 162.4 mL agua (con R = 0.65)


,Ensayo / Parámetro,Valor Medido,Criterio Normado,Dictamen
0,Relación agua/yeso por espolvoreo (R),0.65 (100 g agua / 153.9 g yeso),"Espolvoreo 120 ± 5 s (EN 13279-2, 4.3.1)",CUMPLE TIEMPO
1,Fraguado Inicial (Cuchillo EN 13279-2),24.5 min,"> 20.0 min (EN 13279-1, B7 ref.)",CONFORME
2,Fraguado Completo (Vicat 300 g / 1 mm),36.0 min,"20.0 a 40.0 min (ASTM C28, referencial sin émb...",CONFORME REF.


---
## 📝 3. Conclusiones Técnicas
*(El estudiante redacta aquí su interpretación sobre si el yeso es apto para tarrajeo o refrentado de especímenes).*